# LOGOS — Inference Notebook
## 42k Checkpoint se text generate karo (no training)
**Steps:**
1. Dataset mein `logos-ckpt` add karo (42k .bin + vocab.bin)
2. Run All karo
3. Output mein generated text dekho

In [ ]:
# CELL 1 — Setup
import os, subprocess, glob, time, re

WORK_DIR = '/kaggle/working/LOGOS'
REPO_URL = 'https://github.com/Vikas8719/LOGOS.git'

# GPU check — agar nvidia-smi nahi mila to GPU enable karo
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=memory.total,name,compute_cap',
                        '--format=csv,noheader,nounits'], capture_output=True, text=True)
    parts = r.stdout.strip().split('\n')[0].split(',')
    VRAM_MB  = int(parts[0].strip())
    GPU_NAME = parts[1].strip()
    GPU_CC   = parts[2].strip().replace('.', '')
    print(f'✅ GPU: {GPU_NAME} | VRAM: {VRAM_MB}MB | CC: sm_{GPU_CC}')
except FileNotFoundError:
    raise SystemExit(
        '❌ GPU nahi mila!\n'
        '   Kaggle mein GPU enable karo:\n'
        '   Right panel → Session options → Accelerator → GPU T4 x1\n'
        '   Phir dobara Run karo.'
    )

D_MODEL, N_LAYERS, N_HEADS, SEQ_LEN = 256, 6, 8, 256
print(f'Config: d={D_MODEL} L={N_LAYERS} H={N_HEADS} seq={SEQ_LEN}')


In [ ]:
# CELL 2 — Checkpoint auto-detect
_possible = [
    '/kaggle/input/datasets/vikasawasthiji/logos-ckpt',
    '/kaggle/input/logos-ckpt',
]
CKPT_DIR = next((p for p in _possible if os.path.exists(p)), None)

if not CKPT_DIR:
    raise SystemExit('❌ logos-ckpt dataset nahi mila! Add karo: Notebook → Add Input → logos-ckpt')

bins = sorted([f for f in glob.glob(f'{CKPT_DIR}/*.bin')
               if 'vocab' not in f.lower()])
CKPT  = bins[-1] if bins else None
VOCAB = f'{CKPT_DIR}/vocab.bin'

if not CKPT:
    raise SystemExit('❌ .bin checkpoint nahi mila!')

ckpt_mb  = os.path.getsize(CKPT) / 1024 / 1024
vocab_ok = os.path.exists(VOCAB)
print(f'✅ Checkpoint : {os.path.basename(CKPT)} ({ckpt_mb:.1f} MB)')
print(f'{"✅" if vocab_ok else "⚠️ "} Vocab      : {VOCAB}')

os.environ['LOGOS_CKPT']  = CKPT
os.environ['LOGOS_VOCAB'] = VOCAB

In [ ]:
# CELL 3 — Clone + Build
if not os.path.exists(WORK_DIR):
    os.system(f'git clone {REPO_URL} {WORK_DIR}')
else:
    os.system(f'git -C {WORK_DIR} fetch origin main && git -C {WORK_DIR} reset --hard origin/main')

os.chdir(WORK_DIR)
print('Latest commits:')
os.system('git log --oneline -3')

# Config patch
GPU_SRC = f'{WORK_DIR}/cuda/train_gpu.cu'
with open(GPU_SRC) as f: src = f.read()
patches = [
    (r'cfg\.d_model\s*=\s*scaled\.d_model;',         f'cfg.d_model     = {D_MODEL};'),
    (r'cfg\.num_layers\s*=\s*scaled\.num_layers;',   f'cfg.num_layers  = {N_LAYERS};'),
    (r'cfg\.num_heads\s*=\s*scaled\.num_heads;',     f'cfg.num_heads   = {N_HEADS};'),
    (r'cfg\.max_seq_len\s*=\s*scaled\.max_seq_len;', f'cfg.max_seq_len = {SEQ_LEN};'),
]
for pat, rep in patches:
    src, n = re.subn(pat, rep, src)
    print(f'  {"✅" if n else "ℹ️ "} {rep.strip()[:60]}')
with open(GPU_SRC, 'w') as f: f.write(src)

# Build
BUILD_LOG = '/kaggle/working/build_log.txt'
os.system('rm -rf build && mkdir -p build')
ret = os.system(f'''
cd {WORK_DIR} &&
cmake -B build -DCMAKE_BUILD_TYPE=Release \
  -DCMAKE_CUDA_ARCHITECTURES="{GPU_CC}" \
  -DCMAKE_CUDA_FLAGS="-O3 --use_fast_math" \
  2>&1 | tail -3 &&
cmake --build build --parallel $(nproc) 2>&1 | tee {BUILD_LOG}
''')

BINARY = f'{WORK_DIR}/build/logos_gpu'
if os.path.exists(BINARY):
    print(f'\n✅ Build OK: {os.path.getsize(BINARY)/1024:.0f} KB')
else:
    os.system(f'tail -30 {BUILD_LOG}')
    raise RuntimeError('❌ Build failed')

In [ ]:
# CELL 4 — Generate!
os.chdir(WORK_DIR)

PROMPTS = [
    'एक बार की बात है',
    'भारत एक महान देश',
    'Once upon a time',
    'The sun was shining',
    'In a small village',
    'विज्ञान और तकनीक',
]

def generate(prompt, max_tokens=80, beams=4, hbar=1.0, top_k=50):
    r = subprocess.run(
        [BINARY, '--generate', CKPT, prompt,
         str(max_tokens), str(beams), str(hbar), str(top_k)],
        capture_output=True, text=True, timeout=60, cwd=WORK_DIR
    )
    out = r.stdout + r.stderr
    for line in out.split('\n'):
        if 'Generated:' in line:
            return line.split('Generated:',1)[-1].strip()
    return out[:400]

print('=' * 60)
print(f'LOGOS | {os.path.basename(CKPT)}')
print('=' * 60)

print('\n📝 Standard Generation (hbar=1.0, beams=4):')
print('-' * 60)
for p in PROMPTS:
    result = generate(p)
    print(f'\nPrompt : "{p}"')
    print(f'Output : {result}')

print('\n\n📊 hbar Comparison (same prompt):')
print('-' * 60)
test_p = 'एक बार की बात है'
for hbar, label in [('0.3','Classical (deterministic)'),
                    ('1.0','Standard  (balanced)'),
                    ('2.0','Quantum   (creative)')]:
    result = generate(test_p, hbar=float(hbar), beams=2)
    print(f'\nhbar={hbar} [{label}]')
    print(f'  {result}')

print('\n' + '=' * 60)
print('Done!')

In [ ]:
# CELL 5 — Raw debug (agar CELL 4 mein output nahi aaya)
r = subprocess.run(
    [BINARY, '--generate', CKPT, 'एक बार', '50', '2', '1.0', '30'],
    capture_output=True, text=True, timeout=60, cwd=WORK_DIR
)
print('STDOUT:', r.stdout[:1000])
print('STDERR:', r.stderr[:500])
print('Return code:', r.returncode)